# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

*Lane: Refresh / Content Opportunity Scoring. Label: page impressions fell more than 20% (last 30d vs prior 30d). Language: observed, measured, directional, decision-support.*

## 1. Ranked actions + reason codes

Every eligible page gets an out-of-fold risk score (from clients the model never saw), a **reason code** written from plain past-window facts a human can check, and an action. Risk rank sets the action tier; reason codes explain it.

| Action | Rule | Meaning |
|---|---|---|
| `REFRESH_REVIEW` | top 20% risk | review content for a refresh |
| `REVIEW_METADATA` | top 20% risk and visible but no clicks | check title/description, not the body |
| `MONITOR` | 50-80th percentile risk | re-check next cycle |
| `PROTECT` | growing and bottom 30% risk | don't touch it |
| `NO_ACTION` | everything else | |

Reason codes: `PAST_DROP_CONTINUING`, `CLICKS_FALLING`, `HIGH_TRAFFIC_AT_STAKE`, `VISIBLE_NO_CLICKS`, `MODEL_RISK_ONLY`.

In [ ]:

import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")
URL = "https://raw.githubusercontent.com/AmaanJilani1/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)

# TARGET only. trend_direction is the label source -> never a feature.
df["y"] = (df["trend_direction"] == "down").astype(int)

# Clean past windows. The 90d total = days 1-30 (last30) + 31-60 (prev30) + 61-90 (old).
# Subtracting gives a window that is strictly BEFORE the label window.
for a, b in [("imp", "impressions"), ("clk", "clicks"), ("ses", "sessions")]:
    df[a + "_old"]  = df[b + "_90d"] - df[b + "_last_30d"] - df[b + "_prev_30d"]   # days 61-90 ago
    df[a + "_prev"] = df[b + "_prev_30d"]                                          # days 31-60 ago

# Eligibility: a page can only "decline" if it had visibility to lose.
FLOOR = 100
el = df[df["imp_prev"] >= FLOOR].copy().reset_index(drop=True)
print(f"all pages: {len(df):,} | eligible (prev30 impressions >= {FLOOR}): {len(el):,} "
      f"| clients: {el.client_id.nunique()} | decline base rate: {el.y.mean():.3f}")


from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Raw columns the features are built from. Everything is from BEFORE the label window.
SOURCES = ["imp_old","imp_prev","clk_old","clk_prev","ses_old","ses_prev","search_volume","competition",
           "cpc","word_count","content_age_days","days_since_last_update","content_type","main_intent"]
FORBIDDEN = {"trend_direction","trend_pct","impressions_last_30d","clicks_last_30d","sessions_last_30d",
             "impressions_90d","clicks_90d","sessions_90d","ctr","avg_position","engagement_rate",
             "scroll_rate","ai_traffic_pct","impression_tier","position_tier","y"}
assert not (set(SOURCES) & FORBIDDEN), "a forbidden column is feeding the features"

def make_features(d):
    X = pd.DataFrame(index=d.index)
    for a in ["imp", "clk", "ses"]:
        X["log_" + a + "_old"]  = np.log1p(d[a + "_old"])
        X["log_" + a + "_prev"] = np.log1p(d[a + "_prev"])
        X[a + "_momentum"] = X["log_" + a + "_prev"] - X["log_" + a + "_old"]   # >0 growing, <0 falling
    X["ctr_old"]  = d.clk_old  / d.imp_old.clip(lower=1)
    X["ctr_prev"] = d.clk_prev / d.imp_prev.clip(lower=1)
    X["ctr_delta"] = X.ctr_prev - X.ctr_old
    X["log_search_volume"] = np.log1p(d.search_volume)
    X["competition"], X["cpc"] = d.competition, d.cpc
    X["word_count"], X["wc_missing"] = d.word_count, d.word_count.isna().astype(int)
    X["content_age_days"] = d.content_age_days
    X["updated_over_30d_ago"] = (d.days_since_last_update > 30).astype(int)  # 0 = updated INSIDE the label window
    X["is_keyword_article"]  = (d.content_type == "keyword article").astype(int)
    X["is_comparison"]       = (d.content_type == "comparison article").astype(int)
    for v in ["informational", "transactional", "commercial", "navigational"]:
        X["intent_" + v] = (d.main_intent == v).astype(int)
    return X

X, y, g = make_features(el), el["y"].values, el["client_id"].values

def precision_at_k(y, s, k):
    return y[np.argsort(-s, kind="stable")[:k]].mean()

def cv_scores(model_fn, X, y, g, split="group", n=5, seed=0):
    """Out-of-fold probabilities. split='group' = client holdout (honest); 'random' = leaky comparison."""
    oof = np.zeros(len(y))
    sp = GroupKFold(n).split(X, y, g) if split == "group" else StratifiedKFold(n, shuffle=True, random_state=seed).split(X, y)
    for tr, te in sp:
        m = model_fn(); m.fit(X.iloc[tr], y[tr]); oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return oof

make_lr  = lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=500))
make_hgb = lambda: HGB(max_iter=200, learning_rate=0.05, max_depth=4, random_state=0)

def report(name, s, y=y):
    return {"model": name, "ROC-AUC": roc_auc_score(y, s), "PR-AUC": average_precision_score(y, s),
            "P@50": precision_at_k(y, s, 50), "P@100": precision_at_k(y, s, 100), "P@500": precision_at_k(y, s, 500)}

# Baselines (no fitting): A = the original ML-07 rule, B = "last period's drop continues"
score_rule = ((el.days_since_last_update >= 180) & (el.impressions_90d >= 500)).astype(float).values * el.impressions_90d.values
score_persist = (-X["imp_momentum"]).values

oof = cv_scores(make_lr, X, y, g, "group")
el["risk_score"] = oof
el["risk_pct"] = pd.Series(oof).rank(pct=True).values

c_drop  = X.imp_momentum < -0.3                         # impressions fell >26% from days 61-90 to 31-60
c_vol   = el.imp_prev >= 2000                           # a lot of traffic at stake
c_click = (X.clk_momentum < -0.3) & (el.clk_old >= 5)   # clicks falling on a page with enough clicks to trust
c_ctr   = (el.imp_prev >= 1000) & (el.clk_prev == 0)    # visible, but earns no clicks
c_grow  = X.imp_momentum > 0.2

def codes(i):
    out = []
    if c_drop.iloc[i]:  out.append("PAST_DROP_CONTINUING")
    if c_click.iloc[i]: out.append("CLICKS_FALLING")
    if c_vol.iloc[i]:   out.append("HIGH_TRAFFIC_AT_STAKE")
    if c_ctr.iloc[i]:   out.append("VISIBLE_NO_CLICKS")
    return "|".join(out) if out else "MODEL_RISK_ONLY"
el["reason_code"] = [codes(i) for i in range(len(el))]

def action(i):
    r, p = el.risk_pct.iloc[i], el.reason_code.iloc[i]
    if r >= 0.80 and "VISIBLE_NO_CLICKS" in p: return "REVIEW_METADATA"
    if r >= 0.80: return "REFRESH_REVIEW"
    if r >= 0.50: return "MONITOR"
    if c_grow.iloc[i] and r < 0.30: return "PROTECT"
    return "NO_ACTION"
el["action"] = [action(i) for i in range(len(el))]
order = {"REFRESH_REVIEW": 0, "REVIEW_METADATA": 1, "MONITOR": 2, "PROTECT": 3, "NO_ACTION": 4}
queue = el.assign(_o=el.action.map(order)).sort_values(["_o", "risk_score", "imp_prev"], ascending=[True, False, False]).drop(columns="_o")

print(f"{len(queue):,} pages scored")
print(queue.head(20)[["content_id", "risk_score", "reason_code", "action", "imp_old", "imp_prev"]].round(3).to_string(index=False))


all pages: 30,000 | eligible (prev30 impressions >= 100): 18,010 | clients: 30 | decline base rate: 0.616


18,010 pages scored
          content_id  risk_score                                reason_code         action  imp_old  imp_prev
content_d8fe64aaac6b       0.972                       PAST_DROP_CONTINUING REFRESH_REVIEW     1651       850
content_e454bc31017a       0.972                       PAST_DROP_CONTINUING REFRESH_REVIEW      801       405
content_faed35c8b372       0.967                       PAST_DROP_CONTINUING REFRESH_REVIEW      423       238
content_900f828b925b       0.966                            MODEL_RISK_ONLY REFRESH_REVIEW      738       927
content_aa6cb9d42da5       0.966                            MODEL_RISK_ONLY REFRESH_REVIEW      881       667
content_a6223ddfe9ae       0.966                       PAST_DROP_CONTINUING REFRESH_REVIEW     2037       706
content_9966666373a0       0.960                       PAST_DROP_CONTINUING REFRESH_REVIEW     1764       465
content_b23fa9e12c1d       0.958                       PAST_DROP_CONTINUING REFRESH_REVIEW      910 

## 2. Intended use and limits

**Who uses it:** a content team with limited hours choosing which pages to look at first. **For what:** ordering a review queue, not deciding what to change. **Valid for:** pages with at least 100 impressions in the prior 30 days, similar to the 30 anonymized clients here. **Stops being valid when:** a page had no recent visibility, a client has a very different content mix, Search Console reporting changes, or the data is older than about one cycle (the signals are 30-day windows). The score is a ranking, not a probability: the top decile is over-confident (predicted ~0.88, observed ~0.74).

In [ ]:

print("Evidence the tiers carry information (observed decline rate by action; base rate "
      f"{y.mean():.3f}):")
t = queue.groupby("action").y.agg(["size", "mean"]).rename(columns={"mean": "decline_rate"}).loc[lambda d: d.index.isin(order)]
print(t.sort_index(key=lambda s: s.map(order)).round(3).to_string())
print("\nReason-code frequency:"); print(queue.reason_code.str.split("|").explode().value_counts().to_string())


Evidence the tiers carry information (observed decline rate by action; base rate 0.616):
                 size  decline_rate
action                             
REFRESH_REVIEW   3046         0.709
REVIEW_METADATA   557         0.808
MONITOR          5403         0.680
PROTECT          1537         0.450
NO_ACTION        7467         0.551

Reason-code frequency:
reason_code
PAST_DROP_CONTINUING     8142
MODEL_RISK_ONLY          5549
HIGH_TRAFFIC_AT_STAKE    5264
CLICKS_FALLING           2649
VISIBLE_NO_CLICKS         856


## 3. Human review + the no-go list

**A person must check before acting:** (1) the 90-day shape, to rule out a seasonal topic or a one-off spike in the old window; (2) the page's technical health (indexing, canonical, speed), since a drop is not always a content problem; (3) whether the target keyword still has demand; (4) whether the page was already edited recently (about 60% were updated inside the label window).

**Never automate:** rewriting, merging or deleting pages from the score alone; sending any claim to a client that a refresh *caused* a recovery; ranking promises. Nothing here shows causal impact or anything about Google's algorithm.

In [ ]:

recent = (el.days_since_last_update <= 30)
print(f"Pages edited within the last 30 days: {recent.mean():.0%}. Flag them for a human before any refresh.")
print(queue[queue.action == "REFRESH_REVIEW"].assign(edited_recently=recent).edited_recently.value_counts(normalize=True).round(2).to_string())


Pages edited within the last 30 days: 61%. Flag them for a human before any refresh.
edited_recently
False    0.62
True     0.38


## 4. Monitoring / retrain triggers

Re-fit and re-validate (client holdout) when any of these fire: (1) the decline **base rate** in a new snapshot differs from 0.62 by more than 5 points; (2) **precision@100** on newly labeled pages drops below ~0.70 (the observed value is 0.85, base 0.62); (3) **ROC-AUC** falls below ~0.55; (4) a new client joins with a different content mix (check per-client AUC); (5) Search Console definitions or data pipelines change. Otherwise retrain on a fixed cadence (e.g. each quarter).

In [ ]:

def monitor_checks(frame, score_col="risk_score", label_col="y"):
    yy, ss = frame[label_col].values, frame[score_col].values
    out = {"base_rate": yy.mean(), "roc_auc": roc_auc_score(yy, ss), "p_at_100": precision_at_k(yy, ss, 100)}
    out["RETRAIN?"] = bool(abs(out["base_rate"] - 0.62) > 0.05 or out["roc_auc"] < 0.55 or out["p_at_100"] < 0.70)
    return out
print("Current snapshot:", {k: (round(float(v), 3) if not isinstance(v, bool) else v) for k, v in monitor_checks(el).items()})


Current snapshot: {'base_rate': 0.616, 'roc_auc': 0.612, 'p_at_100': 0.85, 'RETRAIN?': False}


## 5. Exports for the paper

Writes the ranked queue and a summary table to `work/outputs/`. In Colab these land in the temporary session folder: download them from the Files panel and commit them to the repo's `work/outputs/` (counts only; no client names, domains or raw exports).

In [ ]:

os.makedirs("work/outputs", exist_ok=True)
cols = ["content_id", "action", "reason_code", "risk_score", "imp_old", "imp_prev"]
queue[cols].round(4).to_csv("work/outputs/refresh_action_queue.csv", index=False)
summary = queue.groupby("action").agg(pages=("y", "size"), decline_rate=("y", "mean"), median_prev30_impressions=("imp_prev", "median")).round(3)
summary.to_csv("work/outputs/action_summary.csv")
print(summary.to_string()); print("\nwrote work/outputs/refresh_action_queue.csv and work/outputs/action_summary.csv")


                 pages  decline_rate  median_prev30_impressions
action                                                         
MONITOR           5403         0.680                      597.0
NO_ACTION         7467         0.551                      874.0
PROTECT           1537         0.450                     1163.0
REFRESH_REVIEW    3046         0.709                      736.0
REVIEW_METADATA    557         0.808                     1742.0

wrote work/outputs/refresh_action_queue.csv and work/outputs/action_summary.csv


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.